# 03 — Feature Engineering

## Project: Sales Anomaly Detection for a Multi-Outlet F&B Business

### Objective

The goal of this notebook is to create meaningful features from the prepared datasets to support anomaly detection.

The previous notebook created clean transaction-level and daily aggregated datasets.

In this notebook, we will focus on engineering features that help detect unusual daily sales behavior at the outlet level.

### Main dataset

The primary dataset used in this notebook is:

`daily_outlet`

This dataset contains one row per outlet per day and includes:

- transaction count
- total revenue
- average transaction value
- basket size
- zero-value transaction count

### Purpose of feature engineering

The purpose of feature engineering is to create context-aware features that allow us to compare each day's behavior against recent historical patterns.

This will enable anomaly detection methods to identify unusual spikes, drops, and deviations.


### Dataset Selection for Feature Engineering

Multiple processed datasets were created during data preparation, but this notebook focuses only on the datasets needed for daily outlet-level anomaly detection.

The main datasets used are:

- `daily_outlet.csv`
- `daily_paid_outlet.csv`

`daily_outlet` is used to engineer transaction volume features per outlet per day.

`daily_paid_outlet` is used to engineer paid revenue features per outlet per day.

Other processed datasets, such as `transactions_prepared_clean`, `paid_transactions_clean`, `zero_value_transactions_clean`, and `daily_business`, are preserved for future analysis paths such as transaction-level anomaly detection, loyalty behavior analysis, and overall business-level monitoring.

This keeps the current feature engineering notebook focused while maintaining reusable prepared data for later work.

In [1]:
import pandas as pd
import numpy as np

# Load processed datasets
daily_outlet = pd.read_csv("../data/processed/daily_outlet.csv")
daily_paid_outlet = pd.read_csv("../data/processed/daily_paid_outlet.csv")

# Convert date to datetime
daily_outlet["date"] = pd.to_datetime(daily_outlet["date"])
daily_paid_outlet["date"] = pd.to_datetime(daily_paid_outlet["date"])

daily_outlet.head()

,date,outlet,transaction_count,total_revenue,avg_transaction_value,total_items,avg_basket_size,zero_value_transactions
0,2025-01-02,SHOP001,82,2007000,24475.609756,94,1.146341,2
1,2025-01-02,SHOP002,51,1239000,24294.117647,64,1.254902,4
2,2025-01-02,SHOP003,58,1370000,23620.689655,66,1.137931,2
3,2025-01-03,SHOP001,101,2591000,25653.465347,122,1.207921,5
4,2025-01-03,SHOP002,26,540000,20769.230769,26,1.000000,2


In [2]:
daily_outlet = daily_outlet.sort_values(["outlet", "date"]).reset_index(drop=True)

In [3]:
print(daily_outlet.shape)
daily_outlet.head()


(816, 8)


,date,outlet,transaction_count,total_revenue,avg_transaction_value,total_items,avg_basket_size,zero_value_transactions
0,2025-01-02,SHOP001,82,2007000,24475.609756,94,1.146341,2
1,2025-01-03,SHOP001,101,2591000,25653.465347,122,1.207921,5
2,2025-01-04,SHOP001,99,2473000,24979.797980,115,1.161616,3
3,2025-01-05,SHOP001,90,2114000,23488.888889,106,1.177778,3
4,2025-01-06,SHOP001,82,1885000,22987.804878,90,1.097561,2


In [5]:
# Make sure data is sorted correctly for each outlet
daily_outlet = daily_outlet.sort_values(["outlet", "date"]).reset_index(drop=True)

# Calendar features
daily_outlet["day_name"] = daily_outlet["date"].dt.day_name()
daily_outlet["day_of_week"] = daily_outlet["date"].dt.dayofweek  # Mon=0 ... Sun=6
daily_outlet["is_weekend"] = daily_outlet["day_of_week"].isin([5, 6]).astype(int)
daily_outlet["month"] = daily_outlet["date"].dt.month

daily_outlet[["date", "outlet", "day_name", "day_of_week", "is_weekend", "month"]].head()


,date,outlet,day_name,day_of_week,is_weekend,month
0,2025-01-02,SHOP001,Thursday,3,0,1
1,2025-01-03,SHOP001,Friday,4,0,1
2,2025-01-04,SHOP001,Saturday,5,1,1
3,2025-01-05,SHOP001,Sunday,6,1,1
4,2025-01-06,SHOP001,Monday,0,0,1


In [6]:
# Ensure correct sorting before lag/rolling
daily_outlet = daily_outlet.sort_values(["outlet", "date"]).reset_index(drop=True)

# Lag features (previous day)
daily_outlet["tx_lag_1"] = daily_outlet.groupby("outlet")["transaction_count"].shift(1)
daily_outlet["rev_lag_1"] = daily_outlet.groupby("outlet")["total_revenue"].shift(1)

In [7]:
daily_outlet[["date", "outlet", "transaction_count", "tx_lag_1", "total_revenue", "rev_lag_1"]].head(10)

,date,outlet,transaction_count,tx_lag_1,total_revenue,rev_lag_1
0,2025-01-02,SHOP001,82,NaN,2007000,NaN
1,2025-01-03,SHOP001,101,82.0,2591000,2007000.0
2,2025-01-04,SHOP001,99,101.0,2473000,2591000.0
3,2025-01-05,SHOP001,90,99.0,2114000,2473000.0
4,2025-01-06,SHOP001,82,90.0,1885000,2114000.0
5,2025-01-07,SHOP001,96,82.0,2472000,1885000.0
6,2025-01-08,SHOP001,110,96.0,2646000,2472000.0
7,2025-01-09,SHOP001,72,110.0,1959000,2646000.0
8,2025-01-10,SHOP001,90,72.0,2113000,1959000.0
9,2025-01-11,SHOP001,77,90.0,1931000,2113000.0


In [15]:
window = 7

daily_outlet["tx_roll_mean_7"] = (
    daily_outlet.groupby("outlet")["transaction_count"]
    .transform(lambda s: s.rolling(window, min_periods=3).mean())
)

daily_outlet["tx_roll_std_7"] = (
    daily_outlet.groupby("outlet")["transaction_count"]
    .transform(lambda s: s.rolling(window, min_periods=3).std())
)

# ✅ Shift baselines by 1 day to use past-only info (avoid leakage)
daily_outlet["tx_roll_mean_7"] = daily_outlet.groupby("outlet")["tx_roll_mean_7"].shift(1)
daily_outlet["tx_roll_std_7"]  = daily_outlet.groupby("outlet")["tx_roll_std_7"].shift(1)

In [16]:
daily_outlet[["date", "outlet", "transaction_count", "tx_roll_mean_7", "tx_roll_std_7"]].head(15)

,date,outlet,transaction_count,tx_roll_mean_7,tx_roll_std_7
0,2025-01-02,SHOP001,82,NaN,NaN
1,2025-01-03,SHOP001,101,NaN,NaN
2,2025-01-04,SHOP001,99,NaN,NaN
3,2025-01-05,SHOP001,90,94.000000,10.440307
4,2025-01-06,SHOP001,82,93.000000,8.755950
5,2025-01-07,SHOP001,96,90.800000,9.038805
6,2025-01-08,SHOP001,110,91.666667,8.358628
7,2025-01-09,SHOP001,72,94.285714,10.307187
8,2025-01-10,SHOP001,90,92.857143,12.707328
9,2025-01-11,SHOP001,77,91.285714,12.202654


In [17]:
eps = 1e-6

daily_outlet["tx_diff_from_roll7"] = daily_outlet["transaction_count"] - daily_outlet["tx_roll_mean_7"]
daily_outlet["tx_zscore_roll7"] = daily_outlet["tx_diff_from_roll7"] / (daily_outlet["tx_roll_std_7"] + eps)

daily_outlet["tx_lag_1"] = daily_outlet.groupby("outlet")["transaction_count"].shift(1)
daily_outlet["tx_pct_change_1d"] = (
    (daily_outlet["transaction_count"] - daily_outlet["tx_lag_1"]) / (daily_outlet["tx_lag_1"] + eps)
)

In [18]:
daily_outlet[[
    "date", "outlet", "transaction_count",
    "tx_roll_mean_7", "tx_roll_std_7",
    "tx_diff_from_roll7", "tx_zscore_roll7",
    "tx_lag_1", "tx_pct_change_1d"
]].head(20)

,date,outlet,transaction_count,tx_roll_mean_7,tx_roll_std_7,tx_diff_from_roll7,tx_zscore_roll7,tx_lag_1,tx_pct_change_1d
0,2025-01-02,SHOP001,82,NaN,NaN,NaN,NaN,NaN,NaN
1,2025-01-03,SHOP001,101,NaN,NaN,NaN,NaN,82.0,0.231707
2,2025-01-04,SHOP001,99,NaN,NaN,NaN,NaN,101.0,-0.019802
3,2025-01-05,SHOP001,90,94.000000,10.440307,-4.000000,-0.383130,99.0,-0.090909
4,2025-01-06,SHOP001,82,93.000000,8.755950,-11.000000,-1.256288,90.0,-0.088889
5,2025-01-07,SHOP001,96,90.800000,9.038805,5.200000,0.575297,82.0,0.170732
6,2025-01-08,SHOP001,110,91.666667,8.358628,18.333333,2.193342,96.0,0.145833
7,2025-01-09,SHOP001,72,94.285714,10.307187,-22.285714,-2.162153,110.0,-0.345455
8,2025-01-10,SHOP001,90,92.857143,12.707328,-2.857143,-0.224842,72.0,0.250000
9,2025-01-11,SHOP001,77,91.285714,12.202654,-14.285714,-1.170705,90.0,-0.144444


In [19]:
# Top spikes by z-score
daily_outlet.sort_values("tx_zscore_roll7", ascending=False).head(10)[
    ["date", "outlet", "transaction_count", "tx_roll_mean_7", "tx_zscore_roll7"]
]


,date,outlet,transaction_count,tx_roll_mean_7,tx_zscore_roll7
150,2025-06-01,SHOP001,138,80.285714,12.622943
663,2025-05-01,SHOP003,55,27.428571,6.832124
265,2025-09-24,SHOP001,108,84.000000,4.865308
378,2025-04-18,SHOP002,43,26.000000,4.249999
681,2025-05-19,SHOP003,75,47.000000,4.205259
38,2025-02-09,SHOP001,118,104.000000,3.697893
241,2025-08-31,SHOP001,120,92.857143,3.611793
694,2025-06-01,SHOP003,88,54.000000,3.603992
164,2025-06-15,SHOP001,135,112.285714,3.580802
30,2025-02-01,SHOP001,111,85.857143,3.365584


In [20]:
daily_outlet[[
    "date", "outlet", "transaction_count",
    "tx_roll_mean_7", "tx_roll_std_7", "tx_zscore_roll7"
]].head(12)


,date,outlet,transaction_count,tx_roll_mean_7,tx_roll_std_7,tx_zscore_roll7
0,2025-01-02,SHOP001,82,NaN,NaN,NaN
1,2025-01-03,SHOP001,101,NaN,NaN,NaN
2,2025-01-04,SHOP001,99,NaN,NaN,NaN
3,2025-01-05,SHOP001,90,94.000000,10.440307,-0.383130
4,2025-01-06,SHOP001,82,93.000000,8.755950,-1.256288
5,2025-01-07,SHOP001,96,90.800000,9.038805,0.575297
6,2025-01-08,SHOP001,110,91.666667,8.358628,2.193342
7,2025-01-09,SHOP001,72,94.285714,10.307187,-2.162153
8,2025-01-10,SHOP001,90,92.857143,12.707328,-0.224842
9,2025-01-11,SHOP001,77,91.285714,12.202654,-1.170705


In [14]:
daily_outlet[[
    "date", "outlet", "transaction_count",
    "tx_roll_mean_7", "tx_roll_std_7",
    "tx_zscore_roll7", "tx_pct_change_1d"
]].head(15)


,date,outlet,transaction_count,tx_roll_mean_7,tx_roll_std_7,tx_zscore_roll7,tx_pct_change_1d
0,2025-01-02,SHOP001,82,NaN,NaN,NaN,NaN
1,2025-01-03,SHOP001,101,NaN,NaN,NaN,0.231707
2,2025-01-04,SHOP001,99,94.000000,10.440307,0.478913,-0.019802
3,2025-01-05,SHOP001,90,93.000000,8.755950,-0.342624,-0.090909
4,2025-01-06,SHOP001,82,90.800000,9.038805,-0.973580,-0.088889
5,2025-01-07,SHOP001,96,91.666667,8.358628,0.518426,0.170732
6,2025-01-08,SHOP001,110,94.285714,10.307187,1.524595,0.145833
7,2025-01-09,SHOP001,72,92.857143,12.707328,-1.641347,-0.345455
8,2025-01-10,SHOP001,90,91.285714,12.202654,-0.105363,0.250000
9,2025-01-11,SHOP001,77,88.142857,12.707328,-0.876884,-0.144444


### Findings — Transaction Count Rolling Features

Rolling transaction count features were created per outlet using a 7-day rolling window.

The rolling baseline was shifted by one day so that each date is compared only against previous days, avoiding leakage from the current day.

The first few rows per outlet contain missing rolling values because there is not enough historical data to calculate a reliable rolling baseline.

The resulting z-score feature, `tx_zscore_roll7`, measures how far each day's transaction count is from the recent outlet-specific baseline.

Positive z-scores indicate higher-than-usual transaction volume, while negative z-scores indicate lower-than-usual transaction volume.

These features will be useful for detecting daily transaction volume spikes and drops.

In [21]:
# Sort first
daily_paid_outlet = daily_paid_outlet.sort_values(["outlet", "date"]).reset_index(drop=True)

# Previous day paid revenue
daily_paid_outlet["rev_lag_1"] = (
    daily_paid_outlet
    .groupby("outlet")["paid_revenue"]
    .shift(1)
)

# Rolling baseline
window = 7

daily_paid_outlet["rev_roll_mean_7"] = (
    daily_paid_outlet
    .groupby("outlet")["paid_revenue"]
    .transform(lambda s: s.rolling(window, min_periods=3).mean())
)

daily_paid_outlet["rev_roll_std_7"] = (
    daily_paid_outlet
    .groupby("outlet")["paid_revenue"]
    .transform(lambda s: s.rolling(window, min_periods=3).std())
)

# Shift rolling baseline to avoid leakage
daily_paid_outlet["rev_roll_mean_7"] = (
    daily_paid_outlet
    .groupby("outlet")["rev_roll_mean_7"]
    .shift(1)
)

daily_paid_outlet["rev_roll_std_7"] = (
    daily_paid_outlet
    .groupby("outlet")["rev_roll_std_7"]
    .shift(1)
)

# Deviation features
eps = 1e-6

daily_paid_outlet["rev_diff_from_roll7"] = (
    daily_paid_outlet["paid_revenue"] - daily_paid_outlet["rev_roll_mean_7"]
)

daily_paid_outlet["rev_zscore_roll7"] = (
    daily_paid_outlet["rev_diff_from_roll7"] /
    (daily_paid_outlet["rev_roll_std_7"] + eps)
)

daily_paid_outlet["rev_pct_change_1d"] = (
    (daily_paid_outlet["paid_revenue"] - daily_paid_outlet["rev_lag_1"]) /
    (daily_paid_outlet["rev_lag_1"] + eps)
)

In [22]:
daily_paid_outlet[[
    "date",
    "outlet",
    "paid_revenue",
    "rev_roll_mean_7",
    "rev_roll_std_7",
    "rev_zscore_roll7",
    "rev_pct_change_1d"
]].head(12)

,date,outlet,paid_revenue,rev_roll_mean_7,rev_roll_std_7,rev_zscore_roll7,rev_pct_change_1d
0,2025-01-02,SHOP001,2007000,NaN,NaN,NaN,NaN
1,2025-01-03,SHOP001,2591000,NaN,NaN,NaN,0.290982
2,2025-01-04,SHOP001,2473000,NaN,NaN,NaN,-0.045542
3,2025-01-05,SHOP001,2114000,2.357000e+06,308797.668385,-0.786923,-0.145168
4,2025-01-06,SHOP001,1885000,2.296250e+06,279880.182697,-1.469379,-0.108325
5,2025-01-07,SHOP001,2472000,2.214000e+06,304261.400772,0.847955,0.311406
6,2025-01-08,SHOP001,2646000,2.257000e+06,291811.583046,1.333052,0.070388
7,2025-01-09,SHOP001,1959000,2.312571e+06,304267.895745,-1.162040,-0.259637
8,2025-01-10,SHOP001,2113000,2.305714e+06,312725.499592,-0.616241,0.078612
9,2025-01-11,SHOP001,1931000,2.237429e+06,291517.213410,-1.051151,-0.086133


In [24]:
feature_df = daily_outlet.merge(
    daily_paid_outlet[
        [
            "date",
            "outlet",
            "paid_revenue",
            "rev_lag_1",
            "rev_roll_mean_7",
            "rev_roll_std_7",
            "rev_diff_from_roll7",
            "rev_zscore_roll7",
            "rev_pct_change_1d"
        ]
    ],
    on=["date", "outlet"],
    how="left"
)

feature_df.head()

,date,outlet,transaction_count,total_revenue,avg_transaction_value,total_items,avg_basket_size,zero_value_transactions,day_name,day_of_week,...,tx_diff_from_roll7,tx_zscore_roll7,tx_pct_change_1d,paid_revenue,rev_lag_1_y,rev_roll_mean_7,rev_roll_std_7,rev_diff_from_roll7,rev_zscore_roll7,rev_pct_change_1d
0,2025-01-02,SHOP001,82,2007000,24475.609756,94,1.146341,2,Thursday,3,...,NaN,NaN,NaN,2007000,NaN,NaN,NaN,NaN,NaN,NaN
1,2025-01-03,SHOP001,101,2591000,25653.465347,122,1.207921,5,Friday,4,...,NaN,NaN,0.231707,2591000,2007000.0,NaN,NaN,NaN,NaN,0.290982
2,2025-01-04,SHOP001,99,2473000,24979.797980,115,1.161616,3,Saturday,5,...,NaN,NaN,-0.019802,2473000,2591000.0,NaN,NaN,NaN,NaN,-0.045542
3,2025-01-05,SHOP001,90,2114000,23488.888889,106,1.177778,3,Sunday,6,...,-4.0,-0.383130,-0.090909,2114000,2473000.0,2357000.0,308797.668385,-243000.0,-0.786923,-0.145168
4,2025-01-06,SHOP001,82,1885000,22987.804878,90,1.097561,2,Monday,0,...,-11.0,-1.256288,-0.088889,1885000,2114000.0,2296250.0,279880.182697,-411250.0,-1.469379,-0.108325


In [25]:
print("daily_outlet shape:", daily_outlet.shape)
print("feature_df shape:", feature_df.shape)

print("Missing paid revenue rows:", feature_df["paid_revenue"].isna().sum())

daily_outlet shape: (816, 19)
feature_df shape: (816, 26)
Missing paid revenue rows: 0


In [26]:
feature_df_scored = feature_df.dropna(
    subset=["tx_zscore_roll7", "rev_zscore_roll7"]
).copy()

print("Original rows:", feature_df.shape[0])
print("Scored rows:", feature_df_scored.shape[0])
print("Dropped rows (insufficient history):", feature_df.shape[0] - feature_df_scored.shape[0])

Original rows: 816
Scored rows: 807
Dropped rows (insufficient history): 9


In [27]:
feature_df.to_csv("../data/processed/daily_outlet_features.csv", index=False)
feature_df_scored.to_csv("../data/processed/daily_outlet_features_scored.csv", index=False)

## Feature Engineering Summary

A daily outlet-level feature table was created by combining:

- calendar context features (weekday, weekend, month),
- transaction volume lag and rolling baseline features,
- transaction volume deviation features (rolling z-score and percent change),
- paid revenue lag and rolling baseline features,
- paid revenue deviation features (rolling z-score and percent change).

### Merge validation

The feature table merge was performed using `date` and `outlet`.

- `daily_outlet` rows: 816  
- `feature_df` rows: 816  
- missing paid revenue rows: 0  

This confirms that the merge did not lose records and all outlet-day combinations were matched.

### Handling rolling-feature NaNs

Rolling baselines require historical data, so the first few days per outlet have missing rolling values.

A scored feature dataset was created by dropping rows where the anomaly scoring signals were missing:

- `tx_zscore_roll7`
- `rev_zscore_roll7`

### Saved outputs

Two feature datasets were saved:

- `daily_outlet_features.csv` (full feature table)
- `daily_outlet_features_scored.csv` (scoring-ready feature table)

These will be used in the next notebook for baseline anomaly detection.